In [15]:
import pandas as pd
import numpy as np
import datetime
import unicodedata
import difflib
import math
from math import ceil

# ============================================================
# FILE PATHS / SETTINGS
# ============================================================
TEAM_FILE = "2023Teams.txt"
GAME_FILE = "2023Games.txt"
MATCHUP_FILE = "Tournament Matchups 2026.csv"

MATCHUP_YEAR = 2023

# Use only games on/before Selection Sunday
USE_GAMES_THROUGH_SELECTION_SUNDAY = True

# Time-weighting settings
segmentWeighting = [0.5, 1, 2]
useWeighting = True

# ============================================================
# avg_spl dictionary
# ============================================================
# Paste your existing avg_spl dictionary here exactly as-is.
# Example:
# avg_spl = {
#     268: 3.487603305785124,
#     30: 3.209366391184573,
#     ...
# }

avg_spl = {
  240: 2.4005524861878453,
  4: 2.408839779005525,
  17: 2.4198895027624308,
  135: 2.422651933701657,
  297: 2.425414364640884,
  14: 2.444751381215469,
  116: 2.447513812154696,
  136: 2.4696132596685083,
  177: 2.475138121546961,
  350: 2.4806629834254146,
  10: 2.483425414364641,
  284: 2.4917127071823204,
  106: 2.494475138121547,
  58: 2.505524861878453,
  301: 2.5110497237569063,
  20: 2.5248618784530388,
  139: 2.5248618784530388,
  167: 2.5248618784530388,
  196: 2.527624309392265,
  300: 2.530386740331492,
  335: 2.530386740331492,
  173: 2.546961325966851,
  76: 2.5497237569060776,
  11: 2.552486187845304,
  176: 2.5552486187845305,
  336: 2.5552486187845305,
  361: 2.5607734806629834,
  170: 2.56353591160221,
  295: 2.56353591160221,
  222: 2.5662983425414363,
  25: 2.574585635359116,
  216: 2.574585635359116,
  318: 2.580110497237569,
  340: 2.580110497237569,
  129: 2.5828729281767955,
  153: 2.5828729281767955,
  163: 2.5828729281767955,
  230: 2.5828729281767955,
  125: 2.585635359116022,
  339: 2.591160220994475,
  61: 2.604972375690608,
  257: 2.604972375690608,
  77: 2.6077348066298343,
  53: 2.610497237569061,
  249: 2.610497237569061,
  122: 2.613259668508287,
  346: 2.613259668508287,
  161: 2.616022099447514,
  283: 2.6215469613259668,
  324: 2.6215469613259668,
  201: 2.62707182320442,
  209: 2.6298342541436464,
  55: 2.6353591160220997,
  128: 2.6353591160220997,
  325: 2.6353591160220997,
  356: 2.6408839779005526,
  329: 2.643646408839779,
  50: 2.649171270718232,
  68: 2.649171270718232,
  103: 2.6519337016574585,
  91: 2.654696132596685,
  156: 2.657458563535912,
  260: 2.657458563535912,
  303: 2.657458563535912,
  172: 2.665745856353591,
  221: 2.6685082872928176,
  317: 2.6712707182320443,
  296: 2.685082872928177,
  223: 2.6878453038674035,
  74: 2.696132596685083,
  164: 2.696132596685083,
  199: 2.696132596685083,
  34: 2.6988950276243093,
  175: 2.6988950276243093,
  258: 2.6988950276243093,
  233: 2.701657458563536,
  35: 2.7044198895027622,
  198: 2.707182320441989,
  67: 2.712707182320442,
  48: 2.7154696132596685,
  95: 2.720994475138121,
  101: 2.720994475138121,
  308: 2.723756906077348,
  278: 2.7375690607734806,
  246: 2.743093922651933,
  26: 2.74585635359116,
  45: 2.748618784530387,
  90: 2.748618784530387,
  226: 2.751381215469613,
  281: 2.75414364640884,
  311: 2.75414364640884,
  338: 2.7596685082872927,
  348: 2.7596685082872927,
  265: 2.7624309392265194,
  89: 2.765193370165746,
  239: 2.781767955801105,
  259: 2.7845303867403315,
  218: 2.787292817679558,
  275: 2.787292817679558,
  292: 2.787292817679558,
  254: 2.7955801104972378,
  252: 2.798342541436464,
  353: 2.798342541436464,
  56: 2.803867403314917,
  82: 2.8121546961325965,
  328: 2.8121546961325965,
  162: 2.820441988950276,
  105: 2.828729281767956,
  271: 2.828729281767956,
  347: 2.8314917127071824,
  187: 2.834254143646409,
  72: 2.839779005524862,
  212: 2.839779005524862,
  98: 2.850828729281768,
  305: 2.850828729281768,
  138: 2.8535911602209945,
  114: 2.856353591160221,
  144: 2.867403314917127,
  127: 2.8701657458563536,
  269: 2.8701657458563536,
  217: 2.878453038674033,
  287: 2.8812154696132595,
  150: 2.883977900552486,
  2: 2.886740331491713,
  3: 2.8922651933701657,
  224: 2.8950276243093924,
  306: 2.903314917127072,
  323: 2.9060773480662982,
  273: 2.908839779005525,
  331: 2.908839779005525,
  126: 2.914364640883978,
  108: 2.9198895027624308,
  313: 2.922651933701657,
  267: 2.925414364640884,
  140: 2.9281767955801103,
  155: 2.9281767955801103,
  264: 2.9392265193370166,
  251: 2.944751381215469,
  29: 2.947513812154696,
  22: 2.9502762430939224,
  256: 2.9502762430939224,
  112: 2.955801104972376,
  146: 2.955801104972376,
  234: 2.955801104972376,
  357: 2.958563535911602,
  97: 2.9613259668508287,
  268: 2.9613259668508287,
  266: 2.9668508287292816,
  37: 2.9696132596685083,
  134: 2.972375690607735,
  174: 2.975138121546961,
  302: 2.975138121546961,
  225: 2.977900552486188,
  188: 2.9861878453038675,
  294: 2.9861878453038675,
  316: 2.9861878453038675,
  220: 2.9889502762430937,
  228: 2.9917127071823204,
  19: 2.9972375690607733,
  107: 2.9972375690607733,
  241: 3.0027624309392267,
  179: 3.005524861878453,
  307: 3.00828729281768,
  314: 3.016574585635359,
  137: 3.0193370165745854,
  358: 3.022099447513812,
  33: 3.030386740331492,
  288: 3.030386740331492,
  75: 3.0359116022099446,
  208: 3.0386740331491717,
  245: 3.0386740331491717,
  321: 3.0386740331491717,
  118: 3.044198895027624,
  79: 3.0497237569060776,
  93: 3.0552486187845305,
  96: 3.0552486187845305,
  215: 3.0552486187845305,
  322: 3.0580110497237567,
  231: 3.06353591160221,
  102: 3.0662983425414363,
  242: 3.0662983425414363,
  244: 3.069060773480663,
  69: 3.0718232044198897,
  363: 3.0718232044198897,
  123: 3.074585635359116,
  282: 3.0773480662983426,
  31: 3.088397790055249,
  227: 3.088397790055249,
  326: 3.088397790055249,
  360: 3.088397790055249,
  277: 3.0939226519337018,
  21: 3.096685082872928,
  272: 3.0994475138121547,
  362: 3.0994475138121547,
  202: 3.116022099447514,
  111: 3.1187845303867405,
  229: 3.1215469613259668,
  132: 3.12707182320442,
  193: 3.138121546961326,
  274: 3.138121546961326,
  270: 3.1408839779005526,
  286: 3.1408839779005526,
  192: 3.146408839779005,
  46: 3.149171270718232,
  94: 3.149171270718232,
  9: 3.1519337016574585,
  343: 3.1519337016574585,
  8: 3.157458563535912,
  312: 3.160220994475138,
  304: 3.1685082872928176,
  232: 3.1712707182320443,
  263: 3.1712707182320443,
  178: 3.176795580110497,
  298: 3.176795580110497,
  121: 3.179558011049724,
  166: 3.179558011049724,
  310: 3.18232044198895,
  7: 3.1878453038674035,
  63: 3.1906077348066297,
  184: 3.1933701657458564,
  83: 3.1988950276243093,
  352: 3.1988950276243093,
  99: 3.2044198895027622,
  113: 3.207182320441989,
  204: 3.207182320441989,
  255: 3.207182320441989,
  42: 3.209944751381216,
  332: 3.209944751381216,
  327: 3.212707182320442,
  148: 3.218232044198895,
  183: 3.218232044198895,
  59: 3.220994475138121,
  359: 3.226519337016575,
  133: 3.232044198895028,
  315: 3.234806629834254,
  88: 3.2403314917127077,
  104: 3.2403314917127077,
  191: 3.2403314917127077,
  36: 3.243093922651933,
  28: 3.24585635359116,
  334: 3.248618784530387,
  78: 3.251381215469613,
  18: 3.25414364640884,
  168: 3.2569060773480665,
  1: 3.2596685082872927,
  73: 3.2624309392265194,
  190: 3.2624309392265194,
  194: 3.2624309392265194,
  51: 3.265193370165746,
  210: 3.265193370165746,
  333: 3.265193370165746,
  152: 3.2679558011049723,
  157: 3.2679558011049723,
  207: 3.2679558011049723,
  243: 3.270718232044199,
  213: 3.273480662983425,
  189: 3.276243093922652,
  293: 3.2790055248618786,
  355: 3.2845303867403315,
  52: 3.287292817679558,
  354: 3.287292817679558,
  81: 3.292817679558011,
  171: 3.298342541436464,
  200: 3.3011049723756907,
  330: 3.3011049723756907,
  337: 3.3011049723756907,
  238: 3.303867403314917,
  41: 3.3093922651933703,
  248: 3.3093922651933703,
  320: 3.314917127071823,
  181: 3.31767955801105,
  71: 3.325966850828729,
  309: 3.325966850828729,
  342: 3.325966850828729,
  195: 3.328729281767956,
  236: 3.334254143646409,
  299: 3.3370165745856357,
  54: 3.339779005524862,
  211: 3.339779005524862,
  182: 3.342541436464088,
  197: 3.348066298342541,
  151: 3.350828729281768,
  186: 3.350828729281768,
  345: 3.350828729281768,
  349: 3.356353591160221,
  60: 3.3646408839779007,
  205: 3.3701657458563536,
  12: 3.3756906077348066,
  158: 3.378453038674033,
  276: 3.3812154696132595,
  165: 3.383977900552486,
  39: 3.386740331491713,
  66: 3.386740331491713,
  80: 3.3950276243093924,
  247: 3.3977900552486187,
  262: 3.3977900552486187,
  285: 3.3977900552486187,
  235: 3.4005524861878453,
  261: 3.4005524861878453,
  40: 3.408839779005525,
  65: 3.411602209944751,
  149: 3.411602209944751,
  85: 3.414364640883978,
  15: 3.4171270718232045,
  86: 3.4281767955801103,
  131: 3.4281767955801103,
  250: 3.4281767955801103,
  319: 3.4281767955801103,
  30: 3.430939226519337,
  100: 3.433701657458564,
  160: 3.43646408839779,
  159: 3.4419889502762437,
  119: 3.455801104972376,
  124: 3.455801104972376,
  27: 3.4806629834254146,
  109: 3.4917127071823204,
  49: 3.5,
  145: 3.5,
  214: 3.5,
  219: 3.5027624309392267,
  141: 3.5110497237569063,
  341: 3.5193370165745854,
  62: 3.522099447513812,
  38: 3.5248618784530388,
  16: 3.5359116022099446,
  87: 3.5359116022099446,
  203: 3.541436464088398,
  290: 3.546961325966851,
  154: 3.5580110497237567,
  344: 3.574585635359116,
  32: 3.5773480662983426,
  64: 3.580110497237569,
  47: 3.5828729281767955,
  5: 3.588397790055249,
  24: 3.588397790055249,
  57: 3.6022099447513813,
  180: 3.607734806629834,
  120: 3.613259668508287,
  291: 3.616022099447514,
  143: 3.618784530386741,
  44: 3.6243093922651934,
  23: 3.6298342541436464,
  6: 3.696132596685083,
  185: 3.704419889502762,
  115: 3.723756906077348,
  206: 3.748618784530387,
  70: 3.751381215469613,
  84: 3.7596685082872927,
  43: 3.76243093922652,
  130: 3.779005524861879,
  253: 3.792817679558011,
  117: 3.7955801104972378,
  142: 3.8011049723756902,
  279: 3.8535911602209945,
  351: 3.8535911602209945,
  13: 3.8646408839779007,
  289: 3.911602209944751,
  92: 3.969613259668508,
  237: 4.041436464088398,
  169: 4.082872928176796,
  280: 4.085635359116022,
  110: 4.477900552486188,
  147: 4.790055248618785,
}

# Simulation settings
N_SIMS = 100000
SHARPNESS_S = 6.15
RANDOM_SEED = 123

OUTPUT_CSV = f"tournament_sim_results_{MATCHUP_YEAR}.csv"

# ============================================================
# DATE HELPERS
# ============================================================
def convertDateToMassey(dateToConvert):
    return (dateToConvert - datetime.datetime(1970, 1, 1)).days + 719528


def get_selection_sunday(year):
    selection_sunday = {
        2002: "03/10/2002",
        2003: "03/16/2003",
        2004: "03/14/2004",
        2005: "03/13/2005",
        2006: "03/12/2006",
        2007: "03/11/2007",
        2008: "03/16/2008",
        2009: "03/15/2009",
        2010: "03/14/2010",
        2011: "03/13/2011",
        2012: "03/11/2012",
        2013: "03/17/2013",
        2014: "03/16/2014",
        2015: "03/15/2015",
        2016: "03/13/2016",
        2017: "03/12/2017",
        2018: "03/11/2018",
        2019: "03/17/2019",
        2020: "03/15/2020",
        2021: "03/14/2021",
        2022: "03/13/2022",
        2023: "03/12/2023",
        2024: "03/17/2024",
        2025: "03/16/2025",
        2026: "03/15/2026",
        2027: "03/14/2027",
        2028: "03/12/2028",
    }

    if year not in selection_sunday:
        raise ValueError(f"No Selection Sunday date configured for year {year}")

    return datetime.datetime.strptime(selection_sunday[year], "%m/%d/%Y")


# ============================================================
# NAME NORMALIZATION / MATCHING
# ============================================================
def normalize_text(s):
    if pd.isna(s):
        return None

    s = str(s).strip()
    s = s.replace("’", "'").replace("‘", "'").replace("`", "'")
    s = unicodedata.normalize("NFKD", s)
    s = s.replace("&", "and")
    s = s.replace(".", "")
    s = s.replace("-", " ")
    s = s.replace("_", " ")
    s = " ".join(s.split()).lower()

    return s


MANUAL_ALIASES = {
     "Fairleigh Dickinson": "F_Dickinson",
     "Kent St.": "Kent",
     "Louisiana Lafayette": "Louisiana",
     "Northern Kentucky": "N_Kentucky",
     "Texas A&M Corpus Chris": "TAM_C._Christi",
     "College of Charleston": "Col_Charleston",
     "Florida Atlantic": "FL_Atlantic",
     "Western Kentucky": "WKU",
     "American": "American_Univ",
     "Nebraska Omaha": "NE_Omaha",
     "SIU Edwardsville": "SIUE",
     "Saint Francis": "St_Francis_PA",
    "Duke": "Duke",
    "Siena": "Siena",
    "Ohio St.": "Ohio_St",
    "Ohio State": "Ohio_St",
    "TCU": "TCU",
    "St. John’s": "St_John's",
    "St. John's": "St_John's",
    "Northern Iowa": "Northern_Iowa",
    "Kansas": "Kansas",
    "Cal Baptist": "Cal_Baptist",
    "Louisville": "Louisville",
    "South Florida": "South_Florida",
    "Michigan St.": "Michigan_St",
    "Michigan State": "Michigan_St",
    "North Dakota St.": "N_Dakota_St",
    "North Dakota State": "N_Dakota_St",
    "UCLA": "UCLA",
    "UCF": "UCF",
    "Connecticut": "Connecticut",
    "UConn": "Connecticut",
    "Furman": "Furman",
    "Florida": "Florida",
    "Prairie View A&M": "Prairie_View",
    "Lehigh": "Lehigh",
    "Clemson": "Clemson",
    "Iowa": "Iowa",
    "Vanderbilt": "Vanderbilt",
    "McNeese St.": "McNeese_St",
    "McNeese State": "McNeese_St",
    "Nebraska": "Nebraska",
    "Troy": "Troy",
    "North Carolina": "North_Carolina",
    "VCU": "VCU",
    "Illinois": "Illinois",
    "Penn": "Penn",
    "Saint Mary’s": "St_Mary's_CA",
    "Saint Mary's": "St_Mary's_CA",
    "St. Mary's": "St_Mary's_CA",
    "Texas A&M": "Texas_A&M",
    "Houston": "Houston",
    "Idaho": "Idaho",
    "Arizona": "Arizona",
    "LIU Brooklyn": "LIU_Brooklyn",
    "Villanova": "Villanova",
    "Utah St.": "Utah_St",
    "Utah State": "Utah_St",
    "Wisconsin": "Wisconsin",
    "High Point": "High_Point",
    "Arkansas": "Arkansas",
    "Hawaii": "Hawaii",
    "BYU": "BYU",
    "Texas": "Texas",
    "North Carolina St.": "NC_State",
    "NC State": "NC_State",
    "Gonzaga": "Gonzaga",
    "Kennesaw St.": "Kennesaw",
    "Kennesaw State": "Kennesaw",
    "Miami FL": "Miami_FL",
    "Miami (FL)": "Miami_FL",
    "Missouri": "Missouri",
    "Purdue": "Purdue",
    "Queens NC": "Queens_NC",
    "Michigan": "Michigan",
    "UMBC": "UMBC",
    "Howard": "Howard",
    "Georgia": "Georgia",
    "Saint Louis": "St_Louis",
    "St. Louis": "St_Louis",
    "Texas Tech": "Texas_Tech",
    "Akron": "Akron",
    "Alabama": "Alabama",
    "Hofsra": "Hofstra",
    "Hofstra": "Hofstra",
    "Tennessee": "Tennessee",
    "Miami OH": "Miami_OH",
    "Miami (OH)": "Miami_OH",
    "SMU": "SMU",
    "Virginia": "Virginia",
    "Wright St.": "Wright_St",
    "Wright State": "Wright_St",
    "Kentucky": "Kentucky",
    "UC Santa Clara": "Santa_Clara",
    "Santa Clara": "Santa_Clara",
    "Iowa St.": "Iowa_St",
    "Iowa State": "Iowa_St",
    "Tennessee St.": "Tennessee_St",
    "Tennessee State": "Tennessee_St",
}


def resolve_team_name(matchup_name, valid_team_names):
    clean_valid = {str(t).strip() for t in valid_team_names}

    if matchup_name in MANUAL_ALIASES:
        mapped = MANUAL_ALIASES[matchup_name].strip()
        if mapped in clean_valid:
            return mapped

    norm_to_team = {normalize_text(t): str(t).strip() for t in clean_valid}
    norm_name = normalize_text(matchup_name)

    if norm_name in norm_to_team:
        return norm_to_team[norm_name]

    candidates = difflib.get_close_matches(
        norm_name,
        list(norm_to_team.keys()),
        n=1,
        cutoff=0.85
    )

    if candidates:
        return norm_to_team[candidates[0]]

    raise ValueError(f"Could not match matchup team name '{matchup_name}' to team file.")


# ============================================================
# LOAD TEAM DATA
# ============================================================
teamNames = pd.read_csv(TEAM_FILE, header=None, skipinitialspace=True)
teamNames.columns = ["TeamID_1based", "TeamName"]

teamNames["TeamID_1based"] = teamNames["TeamID_1based"].astype(int)
teamNames["TeamName"] = teamNames["TeamName"].astype(str).str.strip()
teamNames["TeamID_0based"] = teamNames["TeamID_1based"] - 1

numTeams = len(teamNames)

# ============================================================
# LOAD GAME DATA
# ============================================================
games = pd.read_csv(GAME_FILE, header=None)
games.columns = [
    "MasseyDay",
    "YYYYMMDD",
    "Team1ID",
    "Team1Loc",
    "Team1Score",
    "Team2ID",
    "Team2Loc",
    "Team2Score",
]

games["Team1ID"] = games["Team1ID"].astype(int) - 1
games["Team2ID"] = games["Team2ID"].astype(int) - 1

if USE_GAMES_THROUGH_SELECTION_SUNDAY:
    selection_sunday_date = get_selection_sunday(MATCHUP_YEAR)
    selection_sunday_int = convertDateToMassey(selection_sunday_date)
    games_for_ratings = games[games["MasseyDay"] <= selection_sunday_int].copy()
else:
    games_for_ratings = games.copy()

if len(games_for_ratings) == 0:
    raise ValueError(
        "No games available for ratings after filtering. "
        "Check MATCHUP_YEAR / game file."
    )

# ============================================================
# LOAD MATCHUP DATA
# ============================================================
matchups = pd.read_csv(MATCHUP_FILE)

if "YEAR" in matchups.columns:
    matchups["YEAR"] = pd.to_numeric(matchups["YEAR"], errors="coerce").astype("Int64")

    available_years = sorted(matchups["YEAR"].dropna().unique().tolist())
    print("Available matchup years:", available_years)

    matchups = matchups[matchups["YEAR"] == MATCHUP_YEAR].copy()
else:
    print("No YEAR column found in matchup CSV. Using full file.")

if len(matchups) == 0:
    raise ValueError(f"No matchup rows found for YEAR={MATCHUP_YEAR} in {MATCHUP_FILE}")

# ============================================================
# IMPORTANT FIX FOR 2025
# ============================================================
# Your 2025 data includes later-round rows too.
# This simulation expects only the opening bracket rows.
# Keep only the Round of 64 rows.
if "CURRENT ROUND" in matchups.columns:
    matchups["CURRENT ROUND"] = pd.to_numeric(
        matchups["CURRENT ROUND"],
        errors="coerce"
    )

    print(
        f"Rows for YEAR={MATCHUP_YEAR} before Round of 64 filter:",
        len(matchups)
    )

    matchups = matchups[matchups["CURRENT ROUND"] == 64].copy()

    print(
        f"Rows for YEAR={MATCHUP_YEAR} after Round of 64 filter:",
        len(matchups)
    )
else:
    print("No CURRENT ROUND column found. Using all rows for selected year.")

if len(matchups) == 0:
    raise ValueError(f"No Round of 64 matchup rows found for YEAR={MATCHUP_YEAR}")

# Optional sanity check.
print("Matchup columns:", list(matchups.columns))
print("Matchup rows used:", len(matchups))

# ============================================================
# EXACT COLLEY-STYLE WEIGHTING
# ============================================================
def weight_avg_spl(x):
    return 3 * math.log10(1 / (avg_spl[x] - 2)) + 1.75


def colley_ranking_with_spl(games_df, num_teams):
    colleyMatrix = 2 * np.diag(np.ones(num_teams))
    b = np.ones(num_teams)

    numGames = len(games_df)
    dayBeforeSeason = games_df.iloc[0]["MasseyDay"] - 1
    lastDayOfSeason = games_df.iloc[numGames - 1]["MasseyDay"]

    for i in range(numGames):
        team1ID = int(games_df.iloc[i]["Team1ID"])
        team1Score = float(games_df.iloc[i]["Team1Score"])
        team2ID = int(games_df.iloc[i]["Team2ID"])
        team2Score = float(games_df.iloc[i]["Team2Score"])
        currentDay = games_df.iloc[i]["MasseyDay"]

        if useWeighting:
            numberSegments = len(segmentWeighting)
            weightIndex = ceil(
                numberSegments
                * ((currentDay - dayBeforeSeason) / (lastDayOfSeason - dayBeforeSeason))
            ) - 1

            weightIndex = max(0, min(weightIndex, numberSegments - 1))
            timeWeight = segmentWeighting[weightIndex]
        else:
            timeWeight = 1

        gameWeight = 1.0

        if team1Score > team2Score:
            if (team2ID + 1) in avg_spl:
                gameWeight = weight_avg_spl(team2ID + 1) * timeWeight
            else:
                gameWeight = timeWeight
        else:
            if (team1ID + 1) in avg_spl:
                gameWeight = weight_avg_spl(team1ID + 1) * timeWeight
            else:
                gameWeight = timeWeight

        colleyMatrix[team1ID, team2ID] -= gameWeight
        colleyMatrix[team2ID, team1ID] -= gameWeight
        colleyMatrix[team1ID, team1ID] += gameWeight
        colleyMatrix[team2ID, team2ID] += gameWeight

        if team1Score > team2Score:
            b[team1ID] += 0.5 * gameWeight
            b[team2ID] -= 0.5 * gameWeight
        elif team2Score > team1Score:
            b[team1ID] -= 0.5 * gameWeight
            b[team2ID] += 0.5 * gameWeight

    ratings = np.linalg.solve(colleyMatrix, b)

    return ratings


ratings = colley_ranking_with_spl(games_for_ratings, numTeams)

id_to_team = dict(zip(teamNames["TeamID_0based"], teamNames["TeamName"]))
team_to_id = dict(zip(teamNames["TeamName"], teamNames["TeamID_0based"]))
team_to_rating = {id_to_team[i]: ratings[i] for i in range(numTeams)}

# ============================================================
# RESOLVE MATCHUP TEAM NAMES
# ============================================================
valid_team_names = set(teamNames["TeamName"].astype(str).str.strip().tolist())

unmatched = []
resolved_teams = []

for x in matchups["TEAM"]:
    try:
        resolved_teams.append(resolve_team_name(x, valid_team_names))
    except ValueError:
        unmatched.append(x)
        resolved_teams.append(None)

if unmatched:
    print("Unmatched matchup names:")
    for name in sorted(set(unmatched)):
        print("  ", name)

    raise ValueError(
        "Some matchup team names could not be resolved. "
        "Add them to MANUAL_ALIASES."
    )

matchups["ResolvedTeam"] = resolved_teams

# ============================================================
# BUILD FIRST-ROUND SLOTS
# ============================================================
rows = matchups["ResolvedTeam"].tolist()
seeds = matchups["SEED"].tolist() if "SEED" in matchups.columns else [None] * len(rows)

first_round_slots = []
i = 0

while i < len(rows):
    # Handles First Four structure:
    # Team A vs Team B
    # Team A vs Team C
    if i + 3 < len(rows) and rows[i] == rows[i + 2] and seeds[i] == seeds[i + 2]:
        first_round_slots.append(([rows[i]], [rows[i + 1], rows[i + 3]]))
        i += 4
    else:
        first_round_slots.append(([rows[i]], [rows[i + 1]]))
        i += 2

if len(first_round_slots) != 32:
    raise ValueError(f"Expected 32 first-round slots, found {len(first_round_slots)}")

quadrant_map = {}

for slot_idx, (left, right) in enumerate(first_round_slots):
    quadrant = slot_idx // 8 + 1

    for t in left + right:
        quadrant_map[t] = quadrant

# ============================================================
# PROBABILITIES
# ============================================================
def model_win_prob(teamA, teamB, sharpness_s=8.0):
    rA = team_to_rating[teamA]
    rB = team_to_rating[teamB]

    return 1.0 / (1.0 + math.exp(-(rA - rB) * sharpness_s))


def simulate_game(teamA, teamB, rng, sharpness_s=8.0):
    p = model_win_prob(teamA, teamB, sharpness_s=sharpness_s)

    return teamA if rng.random() < p else teamB


def pairwise(lst):
    return [(lst[i], lst[i + 1]) for i in range(0, len(lst), 2)]


def simulate_bracket(first_round_slots, rng, sharpness_s=8.0):
    round64_matchups = []
    first_four_winners = []

    for left_options, right_options in first_round_slots:
        left_team = left_options[0]

        if len(right_options) == 1:
            right_team = right_options[0]
        else:
            right_team = simulate_game(
                right_options[0],
                right_options[1],
                rng,
                sharpness_s=sharpness_s
            )

            first_four_winners.append(right_team)

        round64_matchups.append((left_team, right_team))

    round64_teams = [t for game in round64_matchups for t in game]

    round32_teams = [
        simulate_game(a, b, rng, sharpness_s=sharpness_s)
        for a, b in round64_matchups
    ]

    sweet16_teams = [
        simulate_game(a, b, rng, sharpness_s=sharpness_s)
        for a, b in pairwise(round32_teams)
    ]

    elite8_teams = [
        simulate_game(a, b, rng, sharpness_s=sharpness_s)
        for a, b in pairwise(sweet16_teams)
    ]

    final4_teams = [
        simulate_game(a, b, rng, sharpness_s=sharpness_s)
        for a, b in pairwise(elite8_teams)
    ]

    title_game_teams = [
        simulate_game(a, b, rng, sharpness_s=sharpness_s)
        for a, b in pairwise(final4_teams)
    ]

    champion = simulate_game(
        title_game_teams[0],
        title_game_teams[1],
        rng,
        sharpness_s=sharpness_s
    )

    return {
        "FirstFourWin": first_four_winners,
        "Round64": round64_teams,
        "Round32": round32_teams,
        "Sweet16": sweet16_teams,
        "Elite8": elite8_teams,
        "Final4": final4_teams,
        "TitleGame": title_game_teams,
        "Champion": [champion],
    }


# ============================================================
# MONTE CARLO
# ============================================================
rng = np.random.default_rng(RANDOM_SEED)

all_bracket_teams = set()

for left, right in first_round_slots:
    all_bracket_teams.update(left)
    all_bracket_teams.update(right)

counts = {
    team: {
        "FirstFourWin": 0,
        "Round64": 0,
        "Round32": 0,
        "Sweet16": 0,
        "Elite8": 0,
        "Final4": 0,
        "TitleGame": 0,
        "Champion": 0,
    }
    for team in all_bracket_teams
}

checkpoints = [100, 500, 1000, 5000, 10000, N_SIMS]
champion_snapshots = []

for sim in range(1, N_SIMS + 1):
    result = simulate_bracket(first_round_slots, rng, sharpness_s=SHARPNESS_S)

    for round_name, teams_in_round in result.items():
        for team in teams_in_round:
            counts[team][round_name] += 1

    if sim in checkpoints:
        snap = []

        for team in all_bracket_teams:
            snap.append({
                "Simulations": sim,
                "Team": team,
                "ChampionProb": counts[team]["Champion"] / sim
            })

        snap_df = (
            pd.DataFrame(snap)
            .sort_values("ChampionProb", ascending=False)
            .head(12)
        )

        champion_snapshots.append(snap_df)

# ============================================================
# RESULTS TABLE
# ============================================================
results = []

for team in sorted(all_bracket_teams):
    row = {
        "Team": team,
        "Quadrant": quadrant_map.get(team, np.nan),
        "Rating": team_to_rating[team],
    }

    for round_name in [
        "FirstFourWin",
        "Round64",
        "Round32",
        "Sweet16",
        "Elite8",
        "Final4",
        "TitleGame",
        "Champion",
    ]:
        row[f"{round_name}_Count"] = counts[team][round_name]
        row[f"{round_name}_Prob"] = counts[team][round_name] / N_SIMS

    row["WinQuadrant_Prob"] = row["Final4_Prob"]

    row["Convert_R32_to_S16"] = (
        row["Sweet16_Prob"] / row["Round32_Prob"]
        if row["Round32_Prob"] > 0
        else np.nan
    )

    row["Convert_S16_to_E8"] = (
        row["Elite8_Prob"] / row["Sweet16_Prob"]
        if row["Sweet16_Prob"] > 0
        else np.nan
    )

    row["Convert_E8_to_F4"] = (
        row["Final4_Prob"] / row["Elite8_Prob"]
        if row["Elite8_Prob"] > 0
        else np.nan
    )

    row["Convert_F4_to_Title"] = (
        row["TitleGame_Prob"] / row["Final4_Prob"]
        if row["Final4_Prob"] > 0
        else np.nan
    )

    row["Convert_Title_to_Champ"] = (
        row["Champion_Prob"] / row["TitleGame_Prob"]
        if row["TitleGame_Prob"] > 0
        else np.nan
    )

    results.append(row)

results_df = (
    pd.DataFrame(results)
    .sort_values(
        ["Quadrant", "Champion_Prob", "Final4_Prob", "Elite8_Prob"],
        ascending=[True, False, False, False]
    )
    .reset_index(drop=True)
)

results_df.to_csv(OUTPUT_CSV, index=False)

# ============================================================
# PRINT SUMMARY
# ============================================================
print("\n============================================================")
print("TOURNAMENT SIMULATION COMPLETE")
print("============================================================")
print(f"Matchup Year: {MATCHUP_YEAR}")
print(f"Games used for ratings: {len(games_for_ratings)}")
print(f"Bracket teams found: {len(all_bracket_teams)}")
print(f"Simulations: {N_SIMS}")
print(f"Sharpness s: {SHARPNESS_S}")
print(f"avg_spl teams provided: {len(avg_spl)}")
print(f"Output written to: {OUTPUT_CSV}")

print("\nTop 20 teams by title probability")

print(
    results_df.sort_values("Champion_Prob", ascending=False)[[
        "Team",
        "Quadrant",
        "Rating",
        "Round32_Prob",
        "Sweet16_Prob",
        "Elite8_Prob",
        "Final4_Prob",
        "TitleGame_Prob",
        "Champion_Prob",
    ]]
    .head(20)
    .to_string(index=False)
)

for q in sorted(results_df["Quadrant"].dropna().unique()):
    sub = (
        results_df[results_df["Quadrant"] == q]
        .sort_values("Final4_Prob", ascending=False)
    )

    print(f"\n------------------ QUADRANT {int(q)} ------------------")

    print(
        sub[[
            "Team",
            "Rating",
            "Round32_Prob",
            "Sweet16_Prob",
            "Elite8_Prob",
            "Final4_Prob",
            "Champion_Prob",
        ]]
        .to_string(index=False)
    )

print("\nChampion probability convergence snapshots")

for snap in champion_snapshots:
    sim_n = int(snap["Simulations"].iloc[0])

    print(f"\nAfter {sim_n} simulations")

    print(
        snap[["Team", "ChampionProb"]]
        .to_string(index=False)
    )

for q in sorted(results_df["Quadrant"].dropna().unique()):
    qfile = f"quadrant_{int(q)}_results_{MATCHUP_YEAR}.csv"

    (
        results_df[results_df["Quadrant"] == q]
        .sort_values("Champion_Prob", ascending=False)
        .to_csv(qfile, index=False)
    )

    print(f"Saved {qfile}")

print(
    results_df.sort_values("Rating", ascending=False)[[
        "Team",
        "Rating"
    ]]
    .head(25)
)

Available matchup years: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025, 2026]
Rows for YEAR=2023 before Round of 64 filter: 126
Rows for YEAR=2023 after Round of 64 filter: 64
Matchup columns: ['YEAR', 'BY YEAR NO', 'BY ROUND NO', 'TEAM NO', 'TEAM', 'SEED', 'ROUND', 'CURRENT ROUND', 'SCORE']
Matchup rows used: 64

TOURNAMENT SIMULATION COMPLETE
Matchup Year: 2023
Games used for ratings: 5602
Bracket teams found: 64
Simulations: 100000
Sharpness s: 6.15
avg_spl teams provided: 363
Output written to: tournament_sim_results_2023.csv

Top 20 teams by title probability
        Team  Quadrant   Rating  Round32_Prob  Sweet16_Prob  Elite8_Prob  Final4_Prob  TitleGame_Prob  Champion_Prob
     Alabama         1 1.042420       0.93139       0.79182      0.53332      0.39411         0.26367        0.15440
      Kansas         4 1.026716       0.94017       0.81026      0.59559      0.34602         0.21169        0.12522
     Houston         3